# 03 — Indexing and RAG

*Notebook 3 of 4. Continues from `02_evidence_layer.ipynb` — make sure you've run that notebook (or `uv run python scripts/fetch_documents.py` from a terminal) so `data/documents/` has the official PDFs.*

Now we apply parsing + chunking to the full five-PDF corpus, put it in a vector index, and close the first end-to-end RAG loop: retrieve evidence, then generate an answer grounded in it.


In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'data/corpus_manifest.json').exists():
            return candidate
    raise FileNotFoundError(
        'Could not find the workshop repo root (looked for data/corpus_manifest.json in this '
        'directory and its parents). Run this notebook from inside Gravitas_Workshop_Starter.'
    )


ROOT = find_repo_root(Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print('Workshop root:', ROOT)

from config import load_workshop_env
load_workshop_env()  # load .env once, up front, so later @observe spans don't warn about missing keys

from observability.tracing import flush_langfuse, dashboard_base_url


In [ ]:
pdfs = list((ROOT / 'data/documents').glob('*.pdf'))
if not pdfs:
    raise FileNotFoundError(
        'No PDFs found in data/documents/. Run 02_evidence_layer.ipynb first '
        '(or `uv run python scripts/fetch_documents.py` from a terminal) before continuing here.'
    )
print(f'Found {len(pdfs)} PDFs — ready to build the corpus.')


## Mission 6 — Build the processed corpus

Now apply the same parsing + chunking pipeline to all five official PDFs and attach metadata.

This is the point where “a folder of PDFs” becomes **searchable evidence objects**.


📎 **Script reference:** `retrieval/ingestion/corpus.py` (`build_corpus`, `load_corpus`) — runs parsing + chunking across the whole manifest and persists the result to `data/processed/chunks.jsonl`. Run it standalone with `uv run python -m retrieval.ingestion.corpus`.


In [ ]:
from retrieval.ingestion.corpus import build_corpus, load_corpus

# Full-corpus parsing can be the slowest local step (a few minutes for all five PDFs).
# If it stalls badly in the room, ask your facilitator whether a pre-built
# data/processed/chunks.jsonl checkpoint is available, then just run load_corpus() instead.
corpus = build_corpus()
print('Corpus chunks:', len(corpus))
print('Example metadata:', {k: corpus[0][k] for k in ['source_id','company','period','doc_type','page','chunk']})


✅ **Checkpoint:** If a retrieved sentence says “revenue grew 4.2%”, what metadata would you need to show a defensible citation to a user?


## Mission 7 — Indexing and vector search

### 🧠 Concept: semantic search compares meaning

An embedding model maps text into numerical representations. Similar meanings tend to land near one another in that representation space.

We use Pinecone's integrated embedding path so this workshop can focus on retrieval design instead of embedding-service plumbing.

Try the indexing-record exercise below yourself first. Then, whenever you're ready, open a terminal — this uses a working reference implementation, so it doesn't wait on your version above:

```bash
uv run python -m retrieval.ingestion.index_corpus
```


### ✍️ YOUR TURN 1 — Build the record that will enter the vector index
Write `make_pinecone_record(row)` right here: preserve `_id = source_id`, text, source_id, company, period, document type, filename and page.


In [ ]:
def make_pinecone_record(row: dict) -> dict:
    # TODO: return a dict with _id=source_id, plus text, source_id, company, period, doc_type, filename, page
    pass

sample_row = {
    'source_id': 'INFY-FY25-p3-c1', 'text': 'Revenue grew 4.2%.', 'company': 'Infosys',
    'period': 'FY25', 'doc_type': 'results', 'filename': 'infosys.pdf', 'page': 3, 'chunk': 1,
}
record = make_pinecone_record(sample_row)
assert record['_id'] == sample_row['source_id']
for key in ['source_id', 'text', 'company', 'period', 'doc_type', 'filename', 'page']:
    assert record[key] == sample_row[key]
print('make_pinecone_record: looks correct ✅')
record


### Why metadata still matters

Vector similarity answers **“what sounds semantically relevant?”** Metadata answers **“which company / period / document type is even eligible?”**

Good retrieval systems usually use both.


### 🧠 Brief detour — HNSW and IVFFlat
Vector stores usually avoid comparing a query against every vector. **Approximate nearest-neighbour (ANN) indexes** trade a little recall for much faster search.
- **HNSW** builds a multi-layer graph. It usually offers a strong query speed/recall trade-off, but costs more memory and takes longer to build.
- **IVFFlat** partitions vectors into lists/clusters and probes only some of them. It builds faster and uses less memory, but recall depends more on list/probe tuning.
You are **not implementing either index today**. The goal is simply to recognize that vector search itself has an index and a speed/recall trade-off.


In [ ]:
# ✍️ YOUR TURN 2 — Which ANN index fits which constraint?
ann_choices = {
    'Prioritize query quality/speed and can afford more memory/build time': None,  # TODO
    'Prioritize fast builds/lower memory and can tune probes for recall': None,  # TODO
}
ann_choices


## Concept checkpoint — Fine-tuning vs RAG: change behaviour or supply evidence?
**Fine-tuning changes model weights.** It is useful when examples define repeated behavior/style/task patterns.  
**RAG keeps the model fixed and supplies external evidence at inference time.** It is useful when knowledge changes, belongs to private documents, or needs citations/provenance.

| Question | Fine-tuning | RAG |
|---|---|---|
| Shape repeated behaviour/style? | Strong fit | Limited |
| Add tomorrow's filing without retraining? | Poor fit | Strong fit |
| Inherent source citations? | No | It can |
| Main burden | data + training + eval | ingestion + retrieval + eval |

They can be combined.


In [ ]:
# ✍️ YOUR TURN 3 — Decide fine-tuning vs RAG for each scenario.
adaptation_choices = {
    'Answer from a quarterly filing published tomorrow with page citations': None,  # TODO
    'Always classify support tickets into a fixed company taxonomy': None,  # TODO
    'Use private filings AND follow a specialized answer style': None,  # TODO
}
adaptation_choices


## Mission 8 — RAG: retrieve first, then generate

### 🧠 Concept

RAG is not “training the model on your PDFs.” The documents stay outside the model. At question time we:

1. search for relevant chunks,
2. place selected evidence into the model context,
3. ask the model to answer using that evidence.

<img src="../assets/notebook/rag_pipeline.png" width="900" alt="Workshop slide showing the RAG pipeline">


## Concept checkpoint — Why RAG became such a common LLM application pattern
Retrieval + generation has older roots, but the **RAG name and influential formulation arrived in 2020**. It combined a generator with an external retrievable memory and highlighted two limitations of parameter-only knowledge: updating knowledge and showing provenance.
As LLM applications spread, teams wanted private/fresh data without retraining and answers tied to inspectable evidence. Embedding services and vector stores made the pattern practical to ship.

**Advantages:** fresh/private evidence, provenance/citations, modular updates, question-specific context.  
**Disadvantages:** more moving parts; parsing/chunking/retrieval can fail; extra latency/cost; retrieved text can be noisy or malicious; generation can still hallucinate.


In [ ]:
# ✍️ YOUR TURN 4 — Who/what owns fixing each failure?
rag_failure_owner = {
    'A table row disappears during PDF extraction': None,  # TODO
    'The correct chunk exists but is not in top-k': None,  # TODO
    'Correct evidence is supplied but the answer invents another number': None,  # TODO
}
rag_failure_owner


📎 **Script reference:** `retrieval/search/search.py` (`pinecone_search`, `keyword_search`) — dense (Pinecone) and keyword (BM25) search live here as two small, independently testable functions. Run `uv run python -m retrieval.search.search` to see both run standalone.


In [ ]:
from retrieval.search.search import pinecone_search, keyword_search

# ✍️ YOUR TURN 5 — Retrieval has knobs too.
query = 'What FY26 revenue growth guidance did Infosys give?'  # TODO: change later
TOP_K = 5  # TODO: try 3 vs 8 and inspect noise

dense = pinecone_search(query, top_k=TOP_K)
[(x['company'], x['doc_type'], x['page'], x['text'][:180].replace('\n',' ')) for x in dense]


### 🔎 Inspect *before* generating

Do not jump straight to the final answer. Check the retrieved evidence first:

- Is the correct company present?
- Is the correct period present?
- Did the relevant guidance sentence make the top results?
- Are there near-duplicate chunks wasting context?

If retrieval is bad, generation cannot repair missing evidence.


### ✍️ YOUR TURN 6 — Metadata filter before similarity search
If the question explicitly says **Infosys FY26**, there is little value in letting TCS FY25 chunks compete for the first-stage shortlist.
Fill the company/period filter and compare the result list with the unfiltered search above.


In [ ]:
metadata_filter = {
    'company': {'$eq': 'TODO'},  # TODO
    'period': {'$eq': 'TODO'},   # TODO
}
filtered_dense = pinecone_search(query, top_k=TOP_K, filter=metadata_filter)
[(x['company'], x['period'], x['page'], x['text'][:130].replace('\n',' ')) for x in filtered_dense]


### 🔎 Compare the two result lists
Did metadata filtering remove irrelevant companies/periods? Did it accidentally exclude a useful cross-company result?
**Important:** filters improve precision only when the metadata and user intent are both correct. They can also hide evidence if applied too aggressively.


📎 **Script reference:** `retrieval/search/rag.py` (`build_context`) — turns a list of hits into one prompt-ready evidence block. Run it standalone with `uv run python -m retrieval.search.rag`.


In [ ]:
from retrieval.search.rag import build_context

context = build_context(dense)
print(context[:3200])


In [ ]:
from langfuse import observe
from llm.client import call_model, build_messages

# ✍️ YOUR TURN 7 — Close the basic dense-RAG loop.
# Hybrid retrieval comes next; for now keep the path simple: dense search → context → generation.
@observe(name="rag-checkpoint")
def run_rag(question):
    trace_hits = pinecone_search(question, top_k=4)
    trace_context = build_context(trace_hits)
    return None  # TODO: call the same model with build_messages(question, context=trace_context)

grounded_answer = run_rag(query)
print(grounded_answer)
flush_langfuse()


In [ ]:
# ✍️ YOUR TURN 8 — Score the *system*, not the writing style.
rag_check = {
    'retrieved_relevant_evidence': None,      # TODO: True/False
    'answer_uses_supplied_evidence': None,    # TODO: True/False
    'source_ids_are_visible_or_traceable': None,
    'more_inspectable_than_plain_llm': None,
}
rag_check


### 🔭 TRACE CHECKPOINT 2 — RAG

Refresh Langfuse and open the latest `rag-checkpoint` trace.

This time you should see a **small execution tree**:

```text
rag-checkpoint
├─ pinecone-search
└─ workshop-llm-call / generation
```

Answer with a partner:

- Which span selected evidence?
- Which step took longer?
- What changed compared with the plain-LLM trace?
- If the answer were wrong, would you inspect retrieval or generation first—and why?

We intentionally keep this first RAG trace simple. Next we will add keyword retrieval, fusion, and reranking and watch the trace become richer.


✅ **Checkpoint:** RAG mainly fixes the problem **“the model did not have the evidence.”**

It does **not** automatically fix:

- bad parsing,
- bad retrieval,
- wrong document version,
- incorrect reasoning over correct evidence,
- unsupported claims the model adds anyway.

That is why we debug RAG as a pipeline.


---
**Next:** open `04_hybrid_retrieval.ipynb` to add keyword search and reranking on top of this RAG loop.
